In [2]:
import pandas as pd
import numpy as np

# Display settings
pd.set_option("display.max_columns", None)
pd.set_option("display.max_colwidth", None)

# Load datasets
providers = pd.read_csv("../data/raw_data/providers_data.csv")
receivers = pd.read_csv("../data/raw_data/receivers_data.csv")
food_listings = pd.read_csv("../data/raw_data/food_listings_data.csv")
claims = pd.read_csv("../data/raw_data/claims_data.csv")

In [3]:
def basic_eda(df, name):
    print("=" * 80)
    print(f"DATASET: {name}")
    print("=" * 80)

    print("\nShape:")
    print(df.shape)

    print("\nColumns:")
    print(df.columns.tolist())

    print("\nData Types:")
    print(df.dtypes)

    print("\nFirst 5 Rows:")
    print(df.head())

    print("\nMissing Values:")
    print(df.isnull().sum())

    print("\nDuplicate Rows:")
    print(df.duplicated().sum())

    print("\nUnique Values:")
    print(df.nunique())

    print("\nSummary Statistics:")
    print(df.describe(include="all"))

In [4]:
basic_eda(providers, "Providers")
basic_eda(receivers, "Receivers")
basic_eda(food_listings, "Food Listings")
basic_eda(claims, "Claims")

DATASET: Providers

Shape:
(1000, 6)

Columns:
['Provider_ID', 'Name', 'Type', 'Address', 'City', 'Contact']

Data Types:
Provider_ID     int64
Name           object
Type           object
Address        object
City           object
Contact        object
dtype: object

First 5 Rows:
   Provider_ID                         Name           Type  \
0            1             Gonzales-Cochran    Supermarket   
1            2  Nielsen, Johnson and Fuller  Grocery Store   
2            3                 Miller-Black    Supermarket   
3            4   Clark, Prince and Williams  Grocery Store   
4            5               Coleman-Farley  Grocery Store   

                                                 Address            City  \
0    74347 Christopher Extensions\nAndreamouth, OK 91839     New Jessica   
1               91228 Hanson Stream\nWelchtown, OR 27136     East Sheena   
2  561 Martinez Point Suite 507\nGuzmanchester, WA 94320  Lake Jesusview   
3         467 Bell Trail Suite 409\nPort

In [5]:
for col in providers.select_dtypes(include="object").columns:
    print(f"\n{col}")
    print(providers[col].value_counts(dropna=False).head(10))


Name
Name
Brown and Sons       4
Williams PLC         3
Smith Inc            3
Miller Ltd           3
Miller Inc           3
Robinson and Sons    2
Davis Ltd            2
Brown Ltd            2
Jackson LLC          2
Alexander PLC        2
Name: count, dtype: int64

Type
Type
Supermarket         262
Grocery Store       256
Restaurant          246
Catering Service    236
Name: count, dtype: int64

Address
Address
74347 Christopher Extensions\nAndreamouth, OK 91839        1
91228 Hanson Stream\nWelchtown, OR 27136                   1
561 Martinez Point Suite 507\nGuzmanchester, WA 94320      1
467 Bell Trail Suite 409\nPort Jesus, IA 61188             1
078 Matthew Creek Apt. 319\nSaraborough, MA 53978          1
1889 Barnes Gateway\nAdamview, ID 87971                    1
1842 Villarreal Shores\nWilliamfort, CT 44529              1
4770 Miller Light Suite 260\nNew Charlesville, AR 97075    1
169 Scott Keys Suite 000\nEast Michelle, HI 57025          1
71458 Mark Field Apt. 252\nNorth A

In [6]:
# Convert string dates to datetime

food_listings["Expiry_Date"] = pd.to_datetime(
    food_listings["Expiry_Date"],
    errors="coerce"
)

claims["Timestamp"] = pd.to_datetime(
    claims["Timestamp"],
    errors="coerce"
)

print(food_listings["Expiry_Date"].dtype)
print(claims["Timestamp"].dtype)

datetime64[ns]
datetime64[ns]


In [7]:
# Provider IDs in food listings
missing_providers = (
    ~food_listings["Provider_ID"].isin(providers["Provider_ID"])
).sum()

# Food IDs in claims
missing_food = (
    ~claims["Food_ID"].isin(food_listings["Food_ID"])
).sum()

# Receiver IDs in claims
missing_receivers = (
    ~claims["Receiver_ID"].isin(receivers["Receiver_ID"])
).sum()

print("Missing Provider IDs:", missing_providers)
print("Missing Food IDs:", missing_food)
print("Missing Receiver IDs:", missing_receivers)

Missing Provider IDs: 0
Missing Food IDs: 0
Missing Receiver IDs: 0


In [8]:
# Food listing features
food_listings["Expiry_Year"] = food_listings["Expiry_Date"].dt.year
food_listings["Expiry_Month"] = food_listings["Expiry_Date"].dt.month_name()
food_listings["Expiry_Day"] = food_listings["Expiry_Date"].dt.day

# Claim features
claims["Claim_Date"] = claims["Timestamp"].dt.date
claims["Claim_Year"] = claims["Timestamp"].dt.year
claims["Claim_Month"] = claims["Timestamp"].dt.month_name()
claims["Claim_Day"] = claims["Timestamp"].dt.day_name()
claims["Claim_Hour"] = claims["Timestamp"].dt.hour

In [9]:
print(food_listings.head())
print()
print(claims.head())

   Food_ID Food_Name  Quantity Expiry_Date  Provider_ID     Provider_Type  \
0        1     Bread        43  2025-03-17          110     Grocery Store   
1        2      Soup        22  2025-03-24          791     Grocery Store   
2        3    Fruits        46  2025-03-28          478  Catering Service   
3        4    Fruits        15  2025-03-16          930        Restaurant   
4        5      Soup        14  2025-03-19          279        Restaurant   

           Location       Food_Type  Meal_Type  Expiry_Year Expiry_Month  \
0  South Kellyville  Non-Vegetarian  Breakfast         2025        March   
1        West James  Non-Vegetarian     Dinner         2025        March   
2       Lake Regina           Vegan  Breakfast         2025        March   
3         Kellytown           Vegan      Lunch         2025        March   
4        Garciaport           Vegan     Dinner         2025        March   

   Expiry_Day  
0          17  
1          24  
2          28  
3          16  


In [10]:
print(food_listings.dtypes)
print()
print(claims.dtypes)

Food_ID                   int64
Food_Name                object
Quantity                  int64
Expiry_Date      datetime64[ns]
Provider_ID               int64
Provider_Type            object
Location                 object
Food_Type                object
Meal_Type                object
Expiry_Year               int32
Expiry_Month             object
Expiry_Day                int32
dtype: object

Claim_ID                int64
Food_ID                 int64
Receiver_ID             int64
Status                 object
Timestamp      datetime64[ns]
Claim_Date             object
Claim_Year              int32
Claim_Month            object
Claim_Day              object
Claim_Hour              int32
dtype: object


In [11]:
food_master = food_listings.merge(
    providers,
    on="Provider_ID",
    how="left",
    suffixes=("", "_provider")
)

food_master.head()

,Food_ID,Food_Name,Quantity,Expiry_Date,Provider_ID,Provider_Type,Location,Food_Type,Meal_Type,Expiry_Year,Expiry_Month,Expiry_Day,Name,Type,Address,City,Contact
0,1,Bread,43,2025-03-17,110,Grocery Store,South Kellyville,Non-Vegetarian,Breakfast,2025,March,17,Figueroa-Soto,Grocery Store,"113 Donna Pass\nPort Michaelchester, NE 39187",South Kellyville,(599)442-0494
1,2,Soup,22,2025-03-24,791,Grocery Store,West James,Non-Vegetarian,Dinner,2025,March,24,Aguilar Group,Grocery Store,"292 Torres Village\nJohnland, IL 09607",West James,(390)257-0518x4479
2,3,Fruits,46,2025-03-28,478,Catering Service,Lake Regina,Vegan,Breakfast,2025,March,28,"Lopez, Roach and Roach",Catering Service,"8319 Brandi Place Suite 155\nJamesbury, NC 87170",Lake Regina,001-785-448-3246x762
3,4,Fruits,15,2025-03-16,930,Restaurant,Kellytown,Vegan,Lunch,2025,March,16,Cannon-Garcia,Restaurant,"18680 Krystal Inlet Apt. 200\nEast Janettown, MO 64497",Kellytown,9421508200
4,5,Soup,14,2025-03-19,279,Restaurant,Garciaport,Vegan,Dinner,2025,March,19,"Greene, Wood and Hernandez",Restaurant,"69618 Jacob Stravenue\nSouth Jacobton, OH 42099",Garciaport,238.551.4657


In [12]:
claims_master = claims.merge(
    receivers,
    on="Receiver_ID",
    how="left",
    suffixes=("", "_receiver")
)

claims_master.head()

,Claim_ID,Food_ID,Receiver_ID,Status,Timestamp,Claim_Date,Claim_Year,Claim_Month,Claim_Day,Claim_Hour,Name,Type,City,Contact
0,1,164,908,Pending,2025-03-05 05:26:00,2025-03-05,2025,March,Wednesday,5,Abigail Crawford,NGO,Lake Shawn,(010)152-8668
1,2,353,391,Cancelled,2025-03-11 10:24:00,2025-03-11,2025,March,Tuesday,10,Regina Munoz,NGO,Lake Clinton,585.955.3149
2,3,626,492,Completed,2025-03-21 00:59:00,2025-03-21,2025,March,Friday,0,Julie Lewis,NGO,South Steven,(557)352-3497x4606
3,4,61,933,Cancelled,2025-03-04 09:08:00,2025-03-04,2025,March,Tuesday,9,Jodi Lee,NGO,West Paulfort,3517962649
4,5,345,229,Pending,2025-03-14 15:17:00,2025-03-14,2025,March,Friday,15,Tina Watkins,Shelter,Bushview,7774795306


In [13]:
dashboard_data = claims_master.merge(
    food_master,
    on="Food_ID",
    how="left"
)

dashboard_data.head()

,Claim_ID,Food_ID,Receiver_ID,Status,Timestamp,Claim_Date,Claim_Year,Claim_Month,Claim_Day,Claim_Hour,Name_x,Type_x,City_x,Contact_x,Food_Name,Quantity,Expiry_Date,Provider_ID,Provider_Type,Location,Food_Type,Meal_Type,Expiry_Year,Expiry_Month,Expiry_Day,Name_y,Type_y,Address,City_y,Contact_y
0,1,164,908,Pending,2025-03-05 05:26:00,2025-03-05,2025,March,Wednesday,5,Abigail Crawford,NGO,Lake Shawn,(010)152-8668,Dairy,22,2025-03-27,432,Restaurant,Carolchester,Vegan,Dinner,2025,March,27,Meyer and Sons,Restaurant,"08209 Eaton Streets Apt. 968\nSouth Laurenside, RI 88383",Carolchester,001-738-662-6338
1,2,353,391,Cancelled,2025-03-11 10:24:00,2025-03-11,2025,March,Tuesday,10,Regina Munoz,NGO,Lake Clinton,585.955.3149,Fruits,5,2025-03-29,418,Restaurant,Deborahfurt,Non-Vegetarian,Lunch,2025,March,29,"Schwartz, Odonnell and Padilla",Restaurant,"23011 Barnett Falls Apt. 524\nWest Cameron, WY 52614",Deborahfurt,955-658-8107x92494
2,3,626,492,Completed,2025-03-21 00:59:00,2025-03-21,2025,March,Friday,0,Julie Lewis,NGO,South Steven,(557)352-3497x4606,Salad,37,2025-03-29,274,Supermarket,New Dawnborough,Non-Vegetarian,Breakfast,2025,March,29,Shields-Moore,Supermarket,"922 Bailey Overpass Suite 885\nNew William, VA 50509",New Dawnborough,9332911651
3,4,61,933,Cancelled,2025-03-04 09:08:00,2025-03-04,2025,March,Tuesday,9,Jodi Lee,NGO,West Paulfort,3517962649,Fruits,33,2025-03-22,243,Catering Service,West Catherine,Vegetarian,Lunch,2025,March,22,Davis-Hurley,Catering Service,"687 Sparks Forks\nEast Patricia, NC 25546",West Catherine,(172)389-0239x87614
4,5,345,229,Pending,2025-03-14 15:17:00,2025-03-14,2025,March,Friday,15,Tina Watkins,Shelter,Bushview,7774795306,Pasta,14,2025-03-20,346,Grocery Store,Thomasville,Vegan,Lunch,2025,March,20,"Phillips, Wolfe and Martin",Grocery Store,Unit 1376 Box 6294\nDPO AE 62072,Thomasville,001-548-413-4962


In [14]:
food_master.to_csv(
    "../data/processed_data/food_master.csv",
    index=False
)

claims_master.to_csv(
    "../data/processed_data/claims_master.csv",
    index=False
)

dashboard_data.to_csv(
    "../data/processed_data/dashboard_data.csv",
    index=False
)

In [15]:
print(dashboard_data.shape)
print(dashboard_data.columns.tolist())
print(dashboard_data.head())
print(dashboard_data.isnull().sum())

(1000, 30)
['Claim_ID', 'Food_ID', 'Receiver_ID', 'Status', 'Timestamp', 'Claim_Date', 'Claim_Year', 'Claim_Month', 'Claim_Day', 'Claim_Hour', 'Name_x', 'Type_x', 'City_x', 'Contact_x', 'Food_Name', 'Quantity', 'Expiry_Date', 'Provider_ID', 'Provider_Type', 'Location', 'Food_Type', 'Meal_Type', 'Expiry_Year', 'Expiry_Month', 'Expiry_Day', 'Name_y', 'Type_y', 'Address', 'City_y', 'Contact_y']
   Claim_ID  Food_ID  Receiver_ID     Status           Timestamp  Claim_Date  \
0         1      164          908    Pending 2025-03-05 05:26:00  2025-03-05   
1         2      353          391  Cancelled 2025-03-11 10:24:00  2025-03-11   
2         3      626          492  Completed 2025-03-21 00:59:00  2025-03-21   
3         4       61          933  Cancelled 2025-03-04 09:08:00  2025-03-04   
4         5      345          229    Pending 2025-03-14 15:17:00  2025-03-14   

   Claim_Year Claim_Month  Claim_Day  Claim_Hour            Name_x   Type_x  \
0        2025       March  Wednesday         

In [16]:
dashboard_data = dashboard_data.rename(columns={
    "Name_x": "Receiver_Name",
    "Type_x": "Receiver_Type",
    "City_x": "Receiver_City",
    "Contact_x": "Receiver_Contact",
    "Name_y": "Provider_Name",
    "Type_y": "Provider_Category",
    "City_y": "Provider_City",
    "Contact_y": "Provider_Contact"
})

dashboard_data.head()

,Claim_ID,Food_ID,Receiver_ID,Status,Timestamp,Claim_Date,Claim_Year,Claim_Month,Claim_Day,Claim_Hour,Receiver_Name,Receiver_Type,Receiver_City,Receiver_Contact,Food_Name,Quantity,Expiry_Date,Provider_ID,Provider_Type,Location,Food_Type,Meal_Type,Expiry_Year,Expiry_Month,Expiry_Day,Provider_Name,Provider_Category,Address,Provider_City,Provider_Contact
0,1,164,908,Pending,2025-03-05 05:26:00,2025-03-05,2025,March,Wednesday,5,Abigail Crawford,NGO,Lake Shawn,(010)152-8668,Dairy,22,2025-03-27,432,Restaurant,Carolchester,Vegan,Dinner,2025,March,27,Meyer and Sons,Restaurant,"08209 Eaton Streets Apt. 968\nSouth Laurenside, RI 88383",Carolchester,001-738-662-6338
1,2,353,391,Cancelled,2025-03-11 10:24:00,2025-03-11,2025,March,Tuesday,10,Regina Munoz,NGO,Lake Clinton,585.955.3149,Fruits,5,2025-03-29,418,Restaurant,Deborahfurt,Non-Vegetarian,Lunch,2025,March,29,"Schwartz, Odonnell and Padilla",Restaurant,"23011 Barnett Falls Apt. 524\nWest Cameron, WY 52614",Deborahfurt,955-658-8107x92494
2,3,626,492,Completed,2025-03-21 00:59:00,2025-03-21,2025,March,Friday,0,Julie Lewis,NGO,South Steven,(557)352-3497x4606,Salad,37,2025-03-29,274,Supermarket,New Dawnborough,Non-Vegetarian,Breakfast,2025,March,29,Shields-Moore,Supermarket,"922 Bailey Overpass Suite 885\nNew William, VA 50509",New Dawnborough,9332911651
3,4,61,933,Cancelled,2025-03-04 09:08:00,2025-03-04,2025,March,Tuesday,9,Jodi Lee,NGO,West Paulfort,3517962649,Fruits,33,2025-03-22,243,Catering Service,West Catherine,Vegetarian,Lunch,2025,March,22,Davis-Hurley,Catering Service,"687 Sparks Forks\nEast Patricia, NC 25546",West Catherine,(172)389-0239x87614
4,5,345,229,Pending,2025-03-14 15:17:00,2025-03-14,2025,March,Friday,15,Tina Watkins,Shelter,Bushview,7774795306,Pasta,14,2025-03-20,346,Grocery Store,Thomasville,Vegan,Lunch,2025,March,20,"Phillips, Wolfe and Martin",Grocery Store,Unit 1376 Box 6294\nDPO AE 62072,Thomasville,001-548-413-4962


In [17]:
dashboard_data.to_csv(
    "../data/processed_data/dashboard_data.csv",
    index=False
)